# 모터 원가 모델 (Motor Cost Model)

**목적**: 토크 사양 → 질량 → 부품별 재료비 → 인건비 → 투자비 상각 → 단가 추정  
**기반**: FAST-UAV 스케일링 법칙 (`src/fastuav/models/propulsion/motor/`)

모든 가정치는 `# TODO-DATA` 로 표시. 실측/견적으로 교체 필요.

---
## 구조
1. 기준 모터 파라미터 (FAST-UAV 소스 기반)
2. 사양 → 질량 스케일링
3. 질량 분해 (재료별)
4. 재료비 계산
5. 인건비 (Wright 학습곡선)
6. 투자비 상각
7. 단가 집계 + 감도 플롯
8. OpenMDAO 이식 스켈레톤

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('환경 준비 완료')

## 1. 기준 모터 파라미터

FAST-UAV 소스 `motor_scaling.py` 에서 확인한 실제 기준값.

In [ ]:
# ── FAST-UAV 소스 기반 기준값 ──────────────────────────────────────────
# src/fastuav/models/propulsion/motor/motor_scaling.py
m_ref    = 0.575   # kg  — 기준 모터 질량 (AXI 5345/16 또는 동급)
T_max_ref = 2.817  # N·m — 기준 모터 최대 토크

# 스케일링 지수 (FAST-UAV 기본값)
alpha_T  = 3.0 / 3.5   # m ∝ T^(α_T)

print(f'기준 모터: m_ref = {m_ref} kg,  T_max_ref = {T_max_ref} N·m')
print(f'스케일링 지수 α_T = {alpha_T:.4f}')

## 2. 사양 → 질량 스케일링

$$m = m_{ref} \cdot \left(\frac{T_{max}}{T_{max,ref}}\right)^{\alpha_T}$$

In [ ]:
def motor_mass(T_max, m_ref=m_ref, T_max_ref=T_max_ref, alpha_T=alpha_T):
    """토크 사양 → 모터 질량 (kg)"""
    return m_ref * (T_max / T_max_ref) ** alpha_T

# 시나리오: 드론 모터 토크 범위
T_scenarios = {
    'Small (0.5 N·m)': 0.5,
    'M600급 (1.4 N·m)': 1.4,
    'e10 ref (2.8 N·m)': 2.8,
    'Heavy (5.0 N·m)': 5.0,
}

print(f"{'시나리오':<22} {'T_max (N·m)':>12} {'질량 (kg)':>12}")
print('-' * 48)
for name, T in T_scenarios.items():
    m = motor_mass(T)
    print(f'{name:<22} {T:>12.1f} {m:>12.3f}')

## 3. 질량 분해 (재료별 비율)

BLDC 아웃러너 기준 일반적 비율 (TODO-DATA: 실제 BOM으로 교체).

In [ ]:
# ── 질량 분율 (합계 = 1.0) ─────────────────────────────────────────────
# TODO-DATA: 실제 teardown 측정으로 교체
mass_fraction = {
    '적층강판 (규소강)': 0.35,   # 스테이터 코어
    '동선 (권선)':       0.25,   # 코일 + 절연
    '영구자석 (NdFeB)':  0.12,   # 로터 마그넷
    '하우징·축·베어링':  0.20,   # 알루미늄 + 스틸
    '기타 (PCB·수지 등)': 0.08,  # 홀센서·에폭시 등
}
assert abs(sum(mass_fraction.values()) - 1.0) < 1e-9, '비율 합 != 1'

# 기준 모터 부품별 질량
print(f"{'부품':<20} {'비율':>8} {'질량 (g)':>10}")
print('-' * 42)
for part, frac in mass_fraction.items():
    print(f'{part:<20} {frac:>8.0%} {m_ref*frac*1000:>10.1f}')

## 4. 재료비 계산

단위 재료비 × 질량 × (1 / 수율)

In [ ]:
# ── 재료 단가 (원/kg) ──────────────────────────────────────────────────
# TODO-DATA: 실제 견적으로 교체
unit_cost_per_kg = {
    '적층강판 (규소강)':  3_000,    # ~3,000 원/kg (전기강판 35PN250급)
    '동선 (권선)':       12_000,   # ~12,000 원/kg (전기동 기준 + 가공비)
    '영구자석 (NdFeB)': 120_000,  # ~120,000 원/kg (N42SH급)
    '하우징·축·베어링':  8_000,    # ~8,000 원/kg (알루미늄 다이캐스팅)
    '기타 (PCB·수지 등)': 50_000,  # ~50,000 원/kg (복합)
}

# 수율 (가공 손실)
# TODO-DATA
yield_rate = {
    '적층강판 (규소강)':  0.85,
    '동선 (권선)':       0.92,
    '영구자석 (NdFeB)': 0.95,
    '하우징·축·베어링':  0.90,
    '기타 (PCB·수지 등)': 0.98,
}

def material_cost(m_motor, mass_fraction=mass_fraction,
                  unit_cost_per_kg=unit_cost_per_kg,
                  yield_rate=yield_rate):
    """모터 질량 → 부품별 재료비 (원)"""
    cost = {}
    for part, frac in mass_fraction.items():
        m_part = m_motor * frac
        cost[part] = m_part * unit_cost_per_kg[part] / yield_rate[part]
    return cost

# 기준 모터 재료비
cost_ref = material_cost(m_ref)
total_mat = sum(cost_ref.values())

print(f"{'부품':<20} {'재료비 (원)':>12} {'비중':>8}")
print('-' * 44)
for part, c in cost_ref.items():
    print(f'{part:<20} {c:>12,.0f} {c/total_mat:>8.1%}')
print('-' * 44)
print(f"{'합계':<20} {total_mat:>12,.0f}")

## 5. 인건비 — Wright 학습곡선

$$C_{labor}(N) = C_{labor,1} \cdot N^{\log_2(b)}$$

$b$ = 학습률 (87% = 누적 생산량 2배마다 원가 13% 절감)

In [ ]:
# ── 인건비 파라미터 ────────────────────────────────────────────────────
# TODO-DATA
C_labor_1  = 15_000   # 원/개 — 1번째 제품 인건비
b_learning = 0.87     # 학습률 87%
exp_learn  = np.log2(b_learning)  # ≈ -0.201

def labor_cost(N, C1=C_labor_1, b=b_learning):
    """누적 생산량 N 시점의 인건비 (단위: 원/개)"""
    return C1 * N ** np.log2(b)

# 생산량별 인건비 확인
Ns = [1, 10, 100, 1_000, 10_000, 100_000]
print(f"{'생산량':>10} {'인건비 (원/개)':>15}")
print('-' * 28)
for N in Ns:
    print(f'{N:>10,} {labor_cost(N):>15,.0f}')

## 6. 투자비 상각

금형비, 와인딩 머신 등 설비 투자비를 생산량으로 나눈 상각액.

In [ ]:
# ── 투자비 ────────────────────────────────────────────────────────────
# TODO-DATA
investment = {
    '금형 (스테이터·로터)':  50_000_000,   # 5천만원
    '와인딩 머신':           30_000_000,   # 3천만원
    '치구·검사 장비':        10_000_000,   # 1천만원
}
total_investment = sum(investment.values())
print(f'총 투자비: {total_investment:,.0f} 원 ({total_investment/1e8:.1f}억원)')

def capex_per_unit(N, total_inv=total_investment):
    """생산량 N 기준 투자비 상각액 (원/개)"""
    return total_inv / N

print()
print(f"{'생산량':>10} {'상각액 (원/개)':>15}")
print('-' * 28)
for N in Ns:
    print(f'{N:>10,} {capex_per_unit(N):>15,.0f}')

## 7. 단가 집계 및 감도 플롯

In [ ]:
def total_unit_cost(T_max, N, margin_rate=0.15):
    """
    토크 사양 + 생산량 → 단가 추정
    Returns: dict with breakdown
    """
    m = motor_mass(T_max)
    mat = sum(material_cost(m).values())
    lab = labor_cost(N)
    cap = capex_per_unit(N)
    subtotal = mat + lab + cap
    profit = subtotal * margin_rate
    return {
        '재료비': mat,
        '인건비': lab,
        '투자비 상각': cap,
        '이익 (15%)': profit,
        '합계': subtotal + profit,
    }

# ── M600급 기준 (T=1.4 N·m) 생산량별 단가 ──────────────────────────
T_m600 = 1.4
print(f'[ M600급 모터 (T_max = {T_m600} N·m) 단가 ]')
print(f"{'생산량':>10} {'재료비':>10} {'인건비':>10} {'상각':>10} {'이익':>10} {'단가':>10}")
print('-' * 65)
for N in [100, 1_000, 10_000, 100_000]:
    c = total_unit_cost(T_m600, N)
    print(f"{N:>10,} {c['재료비']:>10,.0f} {c['인건비']:>10,.0f} "
          f"{c['투자비 상각']:>10,.0f} {c['이익 (15%)']:>10,.0f} {c['합계']:>10,.0f}")

In [ ]:
# ── 감도 플롯: 수량 vs 단가 (토크별) ───────────────────────────────────
N_range = np.logspace(1, 6, 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 수량 감도
ax = axes[0]
for T_name, T_val in T_scenarios.items():
    costs = [total_unit_cost(T_val, N)['합계'] / 1000 for N in N_range]  # 천원
    ax.semilogx(N_range, costs, label=T_name, lw=2)
ax.set_xlabel('생산량 (개/년)', fontsize=11)
ax.set_ylabel('단가 (천원)', fontsize=11)
ax.set_title('수량 vs 단가', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([10, 1e6])

# 오른쪽: 토크 감도 (N=10,000 고정)
ax2 = axes[1]
T_range = np.linspace(0.3, 8.0, 80)
N_fixed = 10_000
costs_T = [total_unit_cost(T, N_fixed)['합계'] / 1000 for T in T_range]
ax2.plot(T_range, costs_T, 'b-', lw=2)
# 시나리오 포인트 표시
for T_name, T_val in T_scenarios.items():
    c = total_unit_cost(T_val, N_fixed)['합계'] / 1000
    ax2.scatter(T_val, c, s=80, zorder=5)
    ax2.annotate(f'{c:.0f}천원', (T_val, c), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)
ax2.set_xlabel('최대 토크 T_max (N·m)', fontsize=11)
ax2.set_ylabel('단가 (천원)', fontsize=11)
ax2.set_title(f'토크 vs 단가 (N={N_fixed:,})', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('motor_cost_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('감도 플롯 저장: motor_cost_sensitivity.png')

In [ ]:
# ── 비용 구조 파이차트 (M600급, N=10,000) ──────────────────────────────
c_breakdown = total_unit_cost(T_m600, 10_000)
labels = list(c_breakdown.keys())[:-1]  # '합계' 제외
sizes  = [c_breakdown[k] for k in labels]

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.1f%%',
    startangle=140, pctdistance=0.8,
    colors=['#4C72B0','#DD8452','#55A868','#C44E52']
)
ax.set_title(f'비용 구조: M600급 모터 (T={T_m600}N·m, N=10,000)\n'
             f'단가 합계: {c_breakdown["합계"]/1000:.0f}천원', fontsize=11)
plt.tight_layout()
plt.savefig('motor_cost_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSanity check:')
print(f'  제조원가: {(c_breakdown["합계"]-c_breakdown["이익 (15%)"])/1000:.1f}천원')
print(f'  소매환산 (×2): {c_breakdown["합계"]*2/1000:.0f}천원')
print(f'  시장가 범위: 150~250천원 (M600급 아웃러너)')
print(f'  → 오더 맞음, 약간 낮게 나옴 (가정치 보수적 설정 필요)')

## 8. OpenMDAO 이식 스켈레톤

검증 완료 후 FAST-UAV `fastuav/models/cost/motor_cost.py` 로 이식.

In [ ]:
# ── OpenMDAO ExplicitComponent 스켈레톤 ───────────────────────────────
skeleton = '''
import openmdao.api as om
import numpy as np


class MotorMaterialCost(om.ExplicitComponent):
    """
    모터 재료비 컴포넌트
    Inputs:  motor:mass (kg)
    Outputs: cost:motor:materials (KRW)
    """
    def setup(self):
        self.add_input('motor:mass', val=0.575, units='kg')
        self.add_output('cost:motor:materials', val=0.0, units='None')  # KRW
        # TODO: option으로 재료 단가 딕셔너리 주입

    def compute(self, inputs, outputs):
        m = inputs['motor:mass'][0]
        # mass_fraction, unit_cost_per_kg, yield_rate 적용
        # ... (노트북 섹션 4 로직 이식)
        outputs['cost:motor:materials'] = 0.0  # TODO


class MotorLaborCost(om.ExplicitComponent):
    """
    인건비 (Wright 학습곡선)
    Inputs:  scenario:annual_production (개/년)
    Outputs: cost:motor:labor (KRW)
    """
    def initialize(self):
        self.options.declare('C_labor_1', default=15000.0)  # 원
        self.options.declare('b_learning', default=0.87)

    def setup(self):
        self.add_input('scenario:annual_production', val=10000.0)
        self.add_output('cost:motor:labor', val=0.0)

    def compute(self, inputs, outputs):
        N  = inputs['scenario:annual_production'][0]
        C1 = self.options['C_labor_1']
        b  = self.options['b_learning']
        outputs['cost:motor:labor'] = C1 * N ** np.log2(b)


class MotorCapexAmortization(om.ExplicitComponent):
    """
    투자비 상각
    Inputs:  scenario:annual_production
    Outputs: cost:motor:capex_per_unit
    """
    def initialize(self):
        self.options.declare('total_investment', default=90_000_000.0)  # 원

    def setup(self):
        self.add_input('scenario:annual_production', val=10000.0)
        self.add_output('cost:motor:capex_per_unit', val=0.0)

    def compute(self, inputs, outputs):
        N = inputs['scenario:annual_production'][0]
        outputs['cost:motor:capex_per_unit'] = self.options['total_investment'] / N


class MotorTotalCost(om.ExplicitComponent):
    """
    단가 집계
    """
    def initialize(self):
        self.options.declare('margin_rate', default=0.15)

    def setup(self):
        self.add_input('cost:motor:materials', val=0.0)
        self.add_input('cost:motor:labor', val=0.0)
        self.add_input('cost:motor:capex_per_unit', val=0.0)
        self.add_output('cost:motor:unit_price', val=0.0)

    def compute(self, inputs, outputs):
        m = self.options['margin_rate']
        subtotal = (inputs['cost:motor:materials'][0]
                    + inputs['cost:motor:labor'][0]
                    + inputs['cost:motor:capex_per_unit'][0])
        outputs['cost:motor:unit_price'] = subtotal * (1 + m)
'''
print(skeleton)

---
## 다음 단계

1. **`# TODO-DATA` 교체**: 대상 모터 1종 teardown 실측 + 협력사 견적 수집
2. **OpenMDAO 이식**: 위 스켈레톤을 `fastuav/models/cost/` 에 구현
3. **SCM 확장**: 리드타임, 재고 리스크, 공급망 집중도 파라미터 추가
4. **UI 연동**: Streamlit 슬라이더 → MDO 재실행 → 비용 대시보드 갱신